In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
os.getcwd()

'/Users/chance/Desktop/Project/LH_home'

In [3]:
folder = os.listdir('./raw')

file = []
for f in folder:
    if "all" in f:
        data = pd.read_csv("./raw/" + f, encoding="cp949", skiprows=15)
        data = data[list(data.columns[:11])]
        file.append(data)
    
all = pd.concat(file)
all = all.dropna(axis=0)
all

,시군구,번지,도로조건,계약면적(㎡),전월세구분,계약년월,계약일,보증금(만원),월세(만원),건축년도,도로명
0,경상남도 진주시 가좌동,1***,25m미만,12.00,월세,201902,16,200,35,2016.0,가좌길74번길
1,경상남도 진주시 가좌동,1***,8m미만,12.00,전세,201901,5,"4,500",0,2016.0,가좌길78번길
4,경상남도 진주시 가좌동,1***,8m미만,14.20,월세,201906,25,300,32,2016.0,가좌길60번길
5,경상남도 진주시 가좌동,1***,12m미만,14.85,월세,201902,20,500,35,2019.0,가좌길64번길
6,경상남도 진주시 가좌동,1***,12m미만,14.85,월세,201908,28,500,35,2019.0,가좌길64번길
...,...,...,...,...,...,...,...,...,...,...,...
3568,경상남도 진주시 호탄동,6**,8m미만,99.00,월세,202203,14,300,50,2003.0,호탄길27번길
3569,경상남도 진주시 호탄동,6**,8m미만,100.00,월세,202202,13,500,50,2003.0,호탄길14번길
3570,경상남도 진주시 호탄동,6**,8m미만,106.76,전세,202205,6,"14,000",0,2018.0,호탄길9번길
3571,경상남도 진주시 호탄동,6**,25m미만,107.76,월세,202210,19,"9,000",10,2004.0,호탄길


In [4]:
all.도로조건.value_counts()

도로조건
12m미만    7750
8m미만     6668
25m미만    1620
25m이상     243
-          80
Name: count, dtype: int64

In [5]:
geo = pd.read_csv("./raw/geocoding.csv")
all["경도"] = geo.X
all["위도"] = geo.Y

# all = all.fillna({'건축년도':all['건축년도'].mode()[0]})
# all = all.fillna({'도로명':all['도로명'].mode()[0]})
# all = all.fillna({'도로조건':all['도로조건'].mode()[0]})

# 도로조건을 정수로 변환
road=[]
for a in all["도로조건"]:
    if a=="-":
        a = "0"
    if a=="25m이상":
        a = "30"
    road.append(int(str(a).split("m")[0]))
all["도로조건"] = road


# 년월 column 생성
year=[]
month=[]
for d in all["계약년월"]:
    day = pd.to_datetime(d, format="%Y%m")
    year.append(day.year)
    month.append(day.month)

all["year"] = year
all["month"] = month

built_day = [pd.to_datetime(d, format="%Y").year for d in all["건축년도"]]
all["건축년도"] = built_day

# 위치 column만들기
all["location"] = all.시군구 + " " + all.도로명

# 계약년월 기준으로 전월세전환율 입력하기
rate = pd.read_csv("./raw/rate.csv", encoding="cp949")
rate = rate.T
rate = rate[3:]

month = []
for r in rate.index:
    new = "".join(r.split("."))
    month.append(int(new))

rate["계약년월"] = month
rate.columns = ["전월세전환율", "계약년월"]
rate.head()

all = pd.merge(all, rate, how="inner", on="계약년월")

keep = [int("".join(str(k).split(","))) for k in all["보증금(만원)"]]
all["보증금(만원)"] = keep


# 전월세전환율 기준으로 보증금+월세의 가치를 전세로 치환하여 value column으로 만들어준다.
value=[round(all["월세(만원)"][i] * 12 / (all["전월세전환율"][i]/100) + all["보증금(만원)"][i]) for i in range(len(all))]
all["전세전환가격"] = value
#all["도로별 전세값평균"] = all.groupby("도로명").전세전환가격.transform("mean")


# 도로명 결측치 제거
indexNames = []
for i in range(len(all["도로명"])):
    if "번길" not in all["도로명"][i]:
        indexNames.append(i)

all.drop(indexNames, inplace=True)

covid = []
for c in all.year:
    if c in [2020, 2021]:
        covid.append(1)
    else:
        covid.append(0)
all["coivd"] = covid

all["price_per_m2"] = all.전세전환가격 / all["계약면적(㎡)"]
all.head(2)

,시군구,번지,도로조건,계약면적(㎡),전월세구분,계약년월,계약일,보증금(만원),월세(만원),건축년도,도로명,경도,위도,year,month,location,전월세전환율,전세전환가격,coivd,price_per_m2
0,경상남도 진주시 가좌동,1***,25,12.0,월세,201902,16,200,35,2016,가좌길74번길,128.105889,35.157466,2019,2,경상남도 진주시 가좌동 가좌길74번길,5.1,8435,0,702.916667
1,경상남도 진주시 가좌동,1***,8,12.0,전세,201901,5,4500,0,2016,가좌길78번길,128.105889,35.157466,2019,1,경상남도 진주시 가좌동 가좌길78번길,5.3,4500,0,375.000000


In [6]:
# 전세 월세 구분 데이터셋
# all = all[all.전월세구분 == "월세"]
# all = pd.DataFrame.drop(all, columns=["전월세구분"], axis=1)

#원핫인코딩
all = pd.get_dummies(all, columns = ["전월세구분"])


all.head(2)

,시군구,번지,도로조건,계약면적(㎡),계약년월,계약일,보증금(만원),월세(만원),건축년도,도로명,...,위도,year,month,location,전월세전환율,전세전환가격,coivd,price_per_m2,전월세구분_월세,전월세구분_전세
0,경상남도 진주시 가좌동,1***,25,12.0,201902,16,200,35,2016,가좌길74번길,...,35.157466,2019,2,경상남도 진주시 가좌동 가좌길74번길,5.1,8435,0,702.916667,True,False
1,경상남도 진주시 가좌동,1***,8,12.0,201901,5,4500,0,2016,가좌길78번길,...,35.157466,2019,1,경상남도 진주시 가좌동 가좌길78번길,5.3,4500,0,375.000000,False,True


In [9]:
all.drop(columns='전월세구분_전세').to_csv('./data/jinju.csv')